## Segundo Parcial — Minería de Datos II
**Equipo:** Lucas Dugo, Cristian Lo Giudice, Rodolfo Berrone, Santiago Ham Saguier, Andrea Romeo.
Data lake con capas medallion: landing → bronze → silver → gold (clase Spark).
Batch para maestros CSV y streaming para eventos JSONL (clase 1); serving en Astra.
Correr en Colab con *Ejecutar todo*.

In [ ]:
!pip -q install pyspark astrapy

## Instalación
SparkSession como punto de entrada al API de DataFrames y Spark SQL (clase Spark).
Configuro sesión local y reduzco verbosidad de logs.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as funciones
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DoubleType, DateType, TimestampType, IntegerType, LongType,
)

# SparkSession: entrada al API de DataFrames y Spark SQL (clase Spark)
spark = (SparkSession.builder
         .appName("parcial2-colab")
         .master("local[*]")  # ejecución local con todos los cores disponibles
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")  # reducir ruido en logs de ejecución
print("Spark listo, version:", spark.version)

## Repo y datos
Landing: zona de ingestión sin transformar (CSV y JSONL del enunciado).
Clono el repo y verifico que exista `datalake/landing`.

In [ ]:
url_repo = "https://github.com/javi2481/mineria-ii-parcial2.git"
carpeta_repo = "/content/mineria-ii-parcial2"

if not os.path.exists(carpeta_repo):
    os.system(f"git clone {url_repo} {carpeta_repo}")

base = carpeta_repo
ruta_landing = os.path.join(base, "datalake", "landing")

if not os.path.exists(ruta_landing):
    raise FileNotFoundError(f"No encuentro landing en: {ruta_landing}")

print("ok, landing en:", ruta_landing)

## Rutas del datalake
Bronze, silver y gold según grado de curación; checkpoints para estado del Structured Streaming (clase Spark).
Defino rutas de cada zona y de los checkpoints.

In [ ]:
ruta_datalake = os.path.join(base, "datalake")
# bronze batch: maestros CSV
ruta_bronze_csv = os.path.join(ruta_datalake, "bronze", "batch")
# bronze stream: eventos JSONL
ruta_bronze_eventos = os.path.join(ruta_datalake, "bronze", "stream", "usage_events")
ruta_landing_eventos = os.path.join(ruta_landing, "usage_events_stream")
# silver: capa limpia y quarantine
ruta_silver_limpio = os.path.join(ruta_datalake, "silver", "usage_events_clean")
ruta_silver_quarantine = os.path.join(ruta_datalake, "silver", "quarantine")
# gold: agregados para serving
ruta_gold = os.path.join(ruta_datalake, "gold", "examen_mineria_II")
# checkpoints del Structured Streaming (clase Spark)
ruta_checkpoint_bronze = os.path.join(base, "checkpoints", "bronze_stream")
ruta_checkpoint_astra = os.path.join(base, "checkpoints", "serving_astra")

## Bronze — clientes
Bronze: ingestión con mínima transformación y schema explícito — sin `inferSchema` (clase Spark).
Ingesto `customers_orgs`, deduplico y persisto en Parquet.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "customers_orgs.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "customers_orgs")

# schema explícito: contrato de columnas y tipos (clase Spark)
esquema_clientes = StructType([
    StructField("org_id", StringType()), StructField("org_name", StringType()),
    StructField("industry", StringType()), StructField("hq_region", StringType()),
    StructField("plan_tier", StringType()), StructField("is_enterprise", BooleanType()),
    StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
    StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
    StructField("nps_score", DoubleType()),
])

df_clientes = spark.read.option("header", True).schema(esquema_clientes).csv(ruta_csv)
# trazabilidad de ingestión en bronze
df_clientes = df_clientes.withColumn("ingest_ts", funciones.current_timestamp())
df_clientes = df_clientes.withColumn("source_file", funciones.input_file_name())
df_clientes = df_clientes.dropDuplicates(["org_id"])  # dedupe por clave natural

df_clientes.write.mode("overwrite").partitionBy("hq_region").parquet(ruta_salida)
print("clientes:", df_clientes.count(), "filas")  # count() dispara la action

## Bronze — usuarios
Procesamiento batch sobre volúmenes completos de maestros (clase 1).
Mismo patrón: `StructType`, metadatos de ingestión y deduplicación por `user_id`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "users.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "users")

# mismo patrón bronze batch que clientes
esquema_usuarios = StructType([
    StructField("user_id", StringType()), StructField("org_id", StringType()),
    StructField("email", StringType()), StructField("role", StringType()),
    StructField("active", BooleanType()), StructField("created_at", DateType()),
    StructField("last_login", DateType()),
])

df_usuarios = spark.read.option("header", True).schema(esquema_usuarios).csv(ruta_csv)
df_usuarios = df_usuarios.withColumn("ingest_ts", funciones.current_timestamp())
df_usuarios = df_usuarios.withColumn("source_file", funciones.input_file_name())
df_usuarios = df_usuarios.dropDuplicates(["user_id"])

df_usuarios.write.mode("overwrite").parquet(ruta_salida)
print("usuarios:", df_usuarios.count(), "filas")

## Bronze — facturación mensual
Otro maestro en batch: tipos fijos antes de silver (clase Spark).
Cargo `billing_monthly.csv` con schema manual y dedupe por `invoice_id`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "billing_monthly.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "billing_monthly")

esquema_facturacion = StructType([
    StructField("invoice_id", StringType()), StructField("org_id", StringType()),
    StructField("month", DateType()), StructField("subtotal", DoubleType()),
    StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
    StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType()),
])

df_facturacion = spark.read.option("header", True).schema(esquema_facturacion).csv(ruta_csv)
df_facturacion = df_facturacion.withColumn("ingest_ts", funciones.current_timestamp())
df_facturacion = df_facturacion.withColumn("source_file", funciones.input_file_name())
df_facturacion = df_facturacion.dropDuplicates(["invoice_id"])

df_facturacion.write.mode("overwrite").partitionBy("month").parquet(ruta_salida)
print("facturacion:", df_facturacion.count(), "filas")

## Bronze batch — muestra
Parquet: formato columnar recomendado para analytics en el data lake (clase Spark).
Verifico la escritura del maestro de clientes.

In [ ]:
# show() es una action: materializa y muestra el resultado (clase Spark)
spark.read.parquet(os.path.join(ruta_bronze_csv, "customers_orgs")).select(
    "org_id", "plan_tier", "ingest_ts"
).show(3)

## Esquema de eventos
Fuentes semi-estructuradas (JSONL) requieren contrato de datos antes de la lectura (clase Spark).
Declaro `StructType` de `usage_events` para el stream.

In [ ]:
# contrato de columnas para JSONL antes del readStream (clase Spark)
esquema_eventos = StructType([
    StructField("event_id", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("org_id", StringType()),
    StructField("resource_id", StringType()),
    StructField("service", StringType()),
    StructField("region", StringType()),
    StructField("metric", StringType()),
    StructField("value", StringType()),  # en bronze queda String; se castea en silver
    StructField("unit", StringType()),
    StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()),
    StructField("genai_tokens", LongType()),
])

## Bronze streaming
Structured Streaming en micro-batches; watermark acota eventos tardíos y checkpoint materializa progreso (clase Spark).
Ingesto JSONL con `readStream`, deduplico por `event_id` y escribo a bronze.

In [ ]:
# limpieza previa para re-ejecución limpia en Colab
!rm -rf "{ruta_bronze_eventos}" "{ruta_checkpoint_bronze}"
os.makedirs(ruta_bronze_eventos, exist_ok=True)

# readStream: fuente de streaming con schema fijo
stream_entrada = (spark.readStream
                  .schema(esquema_eventos)
                  .option("maxFilesPerTrigger", 10)
                  .json(ruta_landing_eventos))

bronze_eventos = stream_entrada.withColumn("ingest_ts", funciones.current_timestamp())
bronze_eventos = bronze_eventos.withColumn("source_file", funciones.input_file_name())
bronze_eventos = bronze_eventos.withWatermark("timestamp", "2 days")  # tolera eventos tardíos
bronze_eventos = bronze_eventos.dropDuplicates(["event_id"])

# writeStream + checkpoint: persiste micro-batches y estado (clase Spark)
query_bronze = (bronze_eventos.writeStream
                .format("parquet")
                .option("path", ruta_bronze_eventos)
                .option("checkpointLocation", ruta_checkpoint_bronze)
                .outputMode("append")
                .trigger(availableNow=True)
                .start())

query_bronze.awaitTermination()

## Conteo en bronze
Validación de volumen e integridad en bronze antes de silver.
Verifico total de eventos e ids únicos.

In [ ]:
eventos_bronze = spark.read.parquet(ruta_bronze_eventos)
total = eventos_bronze.count()  # action: valida volumen en bronze
unicos = eventos_bronze.select("event_id").distinct().count()  # integridad por event_id
print("eventos en bronze:", total, "| ids unicos:", unicos)

## Silver — reglas de calidad
Silver: reglas de calidad y normalización; inválidos a quarantine (medallion, clase Spark).
Implemento R1–R3 y separo capa limpia.

In [ ]:
eventos = spark.read.parquet(ruta_bronze_eventos)

# value venía número o texto en bronze, acá lo paso a double
eventos = eventos.withColumn("value_num", funciones.col("value").cast("double"))
eventos = eventos.withColumn("usage_date", funciones.to_date("timestamp"))

regla1 = funciones.col("event_id").isNull()  # R1
regla2 = funciones.col("cost_usd_increment") < -0.01  # R2
regla3 = funciones.col("value_num").isNotNull() & funciones.col("unit").isNull()  # R3

eventos = eventos.withColumn(
    "dq_rule",
    funciones.when(regla1, funciones.lit("R1_event_id_nulo"))
     .when(regla2, funciones.lit("R2_costo_anomalo"))
     .when(regla3, funciones.lit("R3_unit_nulo_con_value"))
     .otherwise(funciones.lit(None))
)

fila_invalida = regla1 | regla2 | regla3
df_limpio = eventos.filter(~fila_invalida)
df_quarantine = eventos.filter(fila_invalida)

df_quarantine.write.mode("overwrite").parquet(ruta_silver_quarantine)
print("quarantine:", df_quarantine.count(), "filas")
df_quarantine.select("event_id", "cost_usd_increment", "value", "unit", "dq_rule").show(3, truncate=False)

## Silver — enriquecimiento
Enriquecimiento mediante join con dimensiones de clientes.
Uno eventos limpios con `org_name` y `plan_tier`; particiono por `usage_date`.

In [ ]:
# dimensión de organizaciones desde bronze batch
clientes = (spark.read.parquet(os.path.join(ruta_bronze_csv, "customers_orgs"))
              .select("org_id", "org_name", "plan_tier"))

# join left: enriquezco eventos con atributos del maestro
df_silver = df_limpio.join(clientes, on="org_id", how="left")

columnas_silver = [
    "event_id", "timestamp", "usage_date", "org_id", "org_name", "plan_tier",
    "service", "region", "metric", "value_num", "unit",
    "cost_usd_increment", "schema_version", "carbon_kg", "genai_tokens",
]

df_silver.select(*columnas_silver).write.mode("overwrite").partitionBy("usage_date").parquet(ruta_silver_limpio)
print("silver limpio:", df_silver.count(), "filas")

## Gold — resumen de uso diario
Gold: agregados orientados a consumo analítico y serving (clase Spark).
Resumen diario por org, fecha y servicio en `examen_mineria_II`.

In [ ]:
# re-leo silver porque esta celda corre sola en Colab
df_silver = spark.read.parquet(ruta_silver_limpio)

# una fila por org + día + servicio (org_name y plan_tier son dimensiones)
por_org_dia_servicio = df_silver.groupBy(
    "org_id", "org_name", "plan_tier", "usage_date", "service"
)

resumen_diario = por_org_dia_servicio.agg(
    # costo total del día para ese servicio
    funciones.round(funciones.sum("cost_usd_increment"), 4).alias("total_cost_usd"),
    # requests: solo sumo value_num cuando la métrica es "requests"
    funciones.sum(
        funciones.when(funciones.col("metric") == "requests", funciones.col("value_num"))
        .otherwise(0)
    ).cast("long").alias("total_requests"),
    # tokens de genai: si viene nulo lo trato como 0
    funciones.sum(funciones.coalesce(funciones.col("genai_tokens"), funciones.lit(0)))
     .cast("long").alias("total_genai_tokens"),
    # huella de carbono: idem con coalesce
    funciones.round(funciones.sum(funciones.coalesce(funciones.col("carbon_kg"), funciones.lit(0))), 6)
     .alias("total_carbon_kg"),
    # cuántos eventos entraron en este grupo
    funciones.count("*").cast("int").alias("event_count"),
)

# particiono por fecha como en silver (zona curada del datalake)
resumen_diario.write.mode("overwrite").partitionBy("usage_date").parquet(ruta_gold)
print("gold guardado en:", ruta_gold)
print("filas en gold:", resumen_diario.count())
resumen_diario.show(5, truncate=False)

## Token Astra
AstraDB: Cassandra administrada en la nube para serving por clave (clase Cassandra).
Leo el token desde Secret de Colab o variable de entorno.

In [ ]:
try:
    from google.colab import userdata
    token_astra = userdata.get("ASTRA_TOKEN")
except Exception:
    token_astra = os.environ.get("ASTRA_TOKEN", "")

endpoint_astra = "https://f4eaa1f6-6b6b-4be4-84bd-1962a96c9898-us-east-2.apps.astra.datastax.com"
keyspace = "default_keyspace"
nombre_tabla = "examen_mineria_II"

if not token_astra:
    raise ValueError("Falta ASTRA_TOKEN: Secret en Colab o variable de entorno")

## Tabla en Astra
Cassandra modela por particiones; el diseño responde a los patrones de consulta del enunciado (clase Cassandra).
Tabla `examen_mineria_II` con PK `(org_id, usage_date)` y clustering `service`.

In [ ]:
from astrapy import DataAPIClient
from astrapy.info import CreateTableDefinition, ColumnType

cliente = DataAPIClient()
bd = cliente.get_database(endpoint_astra, token=token_astra)

# PK (org_id, usage_date) + clustering service — alineado a consultas del enunciado
definicion = (
    CreateTableDefinition.builder()
    .add_column("org_id", ColumnType.TEXT)
    .add_column("usage_date", ColumnType.DATE)
    .add_column("service", ColumnType.TEXT)
    .add_column("org_name", ColumnType.TEXT)
    .add_column("plan_tier", ColumnType.TEXT)
    .add_column("total_cost_usd", ColumnType.DOUBLE)
    .add_column("total_requests", ColumnType.BIGINT)
    .add_column("total_genai_tokens", ColumnType.BIGINT)
    .add_column("total_carbon_kg", ColumnType.DOUBLE)
    .add_column("event_count", ColumnType.INT)
    .add_partition_by(["org_id", "usage_date"])  # partition key
    .add_partition_sort({"service": 1})  # clustering key
    .build()
)

if nombre_tabla in bd.list_collection_names(keyspace=keyspace):
    bd.drop_collection(nombre_tabla, keyspace=keyspace)

try:
    bd.drop_table(nombre_tabla, keyspace=keyspace)
except Exception:
    pass

tabla_final = bd.create_table(nombre_tabla, definition=definicion, keyspace=keyspace)
print("tabla lista:", nombre_tabla)

## Carga con foreachBatch
`foreachBatch`: action de streaming para escribir cada micro-batch en un sistema externo (clase Spark).
Cargo gold en Astra con checkpoint de serving.

In [ ]:
from astrapy.data_types import DataAPIDate

!rm -rf "{ruta_checkpoint_astra}"
os.makedirs(os.path.dirname(ruta_checkpoint_astra), exist_ok=True)

def cargar_en_astra(batch_df, batch_id):
    # foreachBatch: convierte cada micro-batch del stream en filas para Astra
    filas = []
    for fila in batch_df.collect():
        filas.append({
            "org_id": fila["org_id"],
            "usage_date": DataAPIDate.from_string(str(fila["usage_date"])),
            "service": fila["service"],
            "org_name": fila["org_name"],
            "plan_tier": fila["plan_tier"],
            "total_cost_usd": float(fila["total_cost_usd"] or 0.0),
            "total_requests": int(fila["total_requests"] or 0),
            "total_genai_tokens": int(fila["total_genai_tokens"] or 0),
            "total_carbon_kg": float(fila["total_carbon_kg"] or 0.0),
            "event_count": int(fila["event_count"] or 0),
        })
    if filas:
        tabla_final.insert_many(filas, chunk_size=50, ordered=True, concurrency=1)
    print(f"batch {batch_id}: {len(filas)} filas")

# readStream sobre parquet gold + checkpoint de carga a Astra
(spark.readStream
 .schema(spark.read.parquet(ruta_gold).schema)
 .parquet(ruta_gold)
 .writeStream
 .foreachBatch(cargar_en_astra)
 .option("checkpointLocation", ruta_checkpoint_astra)
 .trigger(availableNow=True)
 .start()
 .awaitTermination())

print("cargadas:", spark.read.parquet(ruta_gold).count(), "filas")

## Consulta #1
Consulta eficiente con partition key completa `(org_id, usage_date)` (clase Cassandra).
Servicios de `org_pbhsahxt` el `2025-08-01` — consulta #1 del enunciado.

In [ ]:
import pandas as pd

org_consulta = "org_pbhsahxt"
fecha_consulta = DataAPIDate.from_string("2025-08-01")

res1 = list(tabla_final.find({"org_id": org_consulta, "usage_date": fecha_consulta}))
df1 = pd.DataFrame(res1)

print(f"consulta 1 — {org_consulta} el 2025-08-01:")
if not df1.empty:
    display(df1[["service", "total_cost_usd", "total_requests", "total_genai_tokens", "event_count"]])
else:
    print("(sin resultados)")

## Consulta #2
Rango de fechas sobre la misma org; filtro en `service` puede requerir `ALLOW FILTERING` en CQL.
Evolución de `compute` entre julio y agosto 2025 — consulta #2 del enunciado.

In [ ]:
res2 = list(tabla_final.find({"org_id": org_consulta}))
df2 = pd.DataFrame(res2)

fecha_desde = DataAPIDate.from_string("2025-07-01")
fecha_hasta = DataAPIDate.from_string("2025-08-31")

if not df2.empty:
    df2 = df2[
        (df2["service"] == "compute")
        & (df2["usage_date"] >= fecha_desde)
        & (df2["usage_date"] <= fecha_hasta)
    ].sort_values("usage_date", key=lambda s: s.map(lambda d: d.to_string()))

print(f"consulta 2 — {org_consulta}, compute jul-ago:")
if not df2.empty:
    display(df2.assign(usage_date=df2["usage_date"].map(str))[
        ["usage_date", "service", "total_cost_usd", "event_count"]
    ])
else:
    print("(sin resultados)")

## Idempotencia del stream
Idempotencia: re-ejecución sin efectos colaterales en volumen (buena práctica ETL, clase Spark).
Segunda corrida del stream bronze; el conteo debe permanecer igual.

In [ ]:
antes = spark.read.parquet(ruta_bronze_eventos).count()

# segunda corrida del mismo stream — validación de idempotencia
stream_entrada2 = (spark.readStream
                   .schema(esquema_eventos)
                   .option("maxFilesPerTrigger", 10)
                   .json(ruta_landing_eventos))

bronze_eventos2 = stream_entrada2.withColumn("ingest_ts", funciones.current_timestamp())
bronze_eventos2 = bronze_eventos2.withColumn("source_file", funciones.input_file_name())
bronze_eventos2 = bronze_eventos2.withWatermark("timestamp", "2 days")
bronze_eventos2 = bronze_eventos2.dropDuplicates(["event_id"])

query_bronze2 = (bronze_eventos2.writeStream
                 .format("parquet")
                 .option("path", ruta_bronze_eventos)
                 .option("checkpointLocation", ruta_checkpoint_bronze)
                 .outputMode("append")
                 .trigger(availableNow=True)
                 .start())
query_bronze2.awaitTermination()

despues = spark.read.parquet(ruta_bronze_eventos).count()
print("bronze antes:", antes, "despues:", despues, "-> igual?", antes == despues)

## Idempotencia de gold
Reprocesar gold en `overwrite` con el mismo `agg` debe conservar filas y totales.
Reescribo `examen_mineria_II` y comparo count y costo total.

In [ ]:
gold_antes = spark.read.parquet(ruta_gold).count()
costo_antes = spark.read.parquet(ruta_gold).agg(
    funciones.round(funciones.sum("total_cost_usd"), 2)
).collect()[0][0]

# mismo agg que en la celda Gold
df_silver = spark.read.parquet(ruta_silver_limpio)
por_org_dia_servicio = df_silver.groupBy(
    "org_id", "org_name", "plan_tier", "usage_date", "service"
)
gold_repro = por_org_dia_servicio.agg(
    funciones.round(funciones.sum("cost_usd_increment"), 4).alias("total_cost_usd"),
    funciones.sum(
        funciones.when(funciones.col("metric") == "requests", funciones.col("value_num"))
        .otherwise(0)
    ).cast("long").alias("total_requests"),
    funciones.sum(funciones.coalesce(funciones.col("genai_tokens"), funciones.lit(0)))
     .cast("long").alias("total_genai_tokens"),
    funciones.round(funciones.sum(funciones.coalesce(funciones.col("carbon_kg"), funciones.lit(0))), 6)
     .alias("total_carbon_kg"),
    funciones.count("*").cast("int").alias("event_count"),
)
gold_repro.write.mode("overwrite").partitionBy("usage_date").parquet(ruta_gold)

gold_despues = spark.read.parquet(ruta_gold).count()
costo_despues = spark.read.parquet(ruta_gold).agg(
    funciones.round(funciones.sum("total_cost_usd"), 2)
).collect()[0][0]

if gold_antes == gold_despues and costo_antes == costo_despues:
    print(f"idempotencia ok — {gold_despues} filas, costo {costo_despues}")
else:
    print("los numeros no coinciden")

## Tamaño en disco
Parquet comprime y particiona: permite comparar volumen entre capas del lake.
Cuento MB de bronze, silver y gold.

In [ ]:
def tamano(carpeta):
    total = 0
    for ruta, _, archivos in os.walk(carpeta):
        for a in archivos:
            total += os.path.getsize(os.path.join(ruta, a))
    return round(total / 1024 / 1024, 2)

print("bronze batch:", tamano(ruta_bronze_csv), "MB")
print("bronze stream:", tamano(ruta_bronze_eventos), "MB")
print("silver:", tamano(os.path.join(ruta_datalake, "silver")), "MB")
print("gold:", tamano(ruta_gold), "MB")